Install Dependencies

In [ ]:
!pip install langchain langchain-community langchain-text-splitters faiss-cpu sentence-transformers gradio pypdf openai

Imports

In [ ]:
import os
import gradio as gr

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

from openai import OpenAI

LLM Setup (OpenRouter)

In [ ]:
from google.colab import userdata
import os

# 🔑 Fetch key securely
os.environ["OPENAI_API_KEY"] = userdata.get("StepUp")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1"
)

def call_llm(prompt):
    response = client.chat.completions.create(
        model="stepfun/step-3.5-flash:free",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )
    return response.choices[0].message.content

DocEater() (Ingestion)

In [ ]:
def DocEater(pdf_file):

    loader = PyPDFLoader(pdf_file.name)
    pages = loader.load()

    if len(pages) < 10:
        return None, None, "PDF must be at least 10 pages"

    full_text = "\n".join([p.page_content for p in pages])

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )

    chunks = splitter.split_documents(pages)

    return chunks, full_text, "Document processed successfully"

MyWIKI() (Vector DB)

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = None

def MyWIKI(chunks):

    global vectorstore

    vectorstore = FAISS.from_documents(chunks, embedding_model)

    return "MyWIKI is Ready!"

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tools

🔹 TellInShort() (Summarizer)

In [ ]:
def TellInShort(full_text):

    prompt = f"""
Provide a clear and structured summary of the document below:

{full_text[:12000]}
"""

    return call_llm(prompt)

🔹 AnswerMe() (QA Tool)

In [ ]:
def AnswerMe(query):

    docs = vectorstore.similarity_search(query, k=4)

    context = "\n\n".join([
        f"(Page {doc.metadata.get('page', 'N/A')}) {doc.page_content}"
        for doc in docs
    ])

    prompt = f"""
Answer ONLY using the context below.
Cite page numbers clearly.

Context:
{context}

Question:
{query}
"""

    return call_llm(prompt)

🔹 ApproachHow() (Approach Tool)

In [ ]:
def ApproachHow(query, full_text):

    prompt = f"""
Using the document below, suggest a clear step-by-step approach:

Document:
{full_text[:12000]}

Task:
{query}
"""

    return call_llm(prompt)

Hero() (Dispatcher)

In [ ]:
def Hero(query, full_text):

    q = query.lower()

    if "summarize" in q or "summary" in q:
        tool_used = "TellInShort()"
        answer = TellInShort(full_text)

    elif "approach" in q:
        tool_used = "ApproachHow()"
        answer = ApproachHow(query, full_text)

    else:
        tool_used = "AnswerMe()"
        answer = AnswerMe(query)

    return f"🛠 Tool Used: {tool_used}\n\n{answer}"

UI()

In [ ]:
chunks_global = None
full_text_global = None

def process_pdf(pdf):

    global chunks_global, full_text_global

    chunks_global, full_text_global, msg = DocEater(pdf)

    if chunks_global is None:
        return msg

    return MyWIKI(chunks_global)


def ask_query(query):

    return Hero(query, full_text_global)


def UI():

    with gr.Blocks() as demo:

        gr.Markdown("# 📄 IntelligentASKER V1.0")

        pdf_input = gr.File(label="Upload PDF")
        upload_btn = gr.Button("Ingest Document")
        upload_output = gr.Textbox(label="Status:")

        query_input = gr.Textbox(label="Ask a question:")
        query_btn = gr.Button("Ask")
        with gr.Group():
          gr.Markdown("### Intelligent Asker Response")
          answer_output = gr.Markdown()

        upload_btn.click(process_pdf, pdf_input, upload_output)
        query_btn.click(ask_query, query_input, answer_output)

    demo.launch()

Testing The Application V 1.0 (if works will move Ahead)

In [ ]:
UI()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a45d7fd618448e0df2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
